# 🏆 R2AI2026 - ViFinQA Text-to-Pandas Pipeline
**Google Colab Free (T4 GPU - 15GB VRAM)**

## Nguyên tắc vàng:
- **Code** → Ổ SSD `/content/Code_Moi` (clone từ GitHub)
- **Data gốc** (ViFinQA) → Google Drive (chỉ đọc)
- **Data sinh ra** (CSV/Metadata/Index) → Ổ SSD `/content/data_output` (tốc độ cao)
- **Kết quả** (submission) → Google Drive (lưu trữ lâu dài)

## Thứ tự chạy:
1. Mount Drive + Clone repo
2. Cài thư viện
3. Kiểm tra GPU
4. Pipeline 1: Ingestion (chỉ chạy 1 lần, ghi lên SSD)
5. Pipeline 2: Query + LLM (tự resume qua checkpoint)
6. Kiểm tra & lưu kết quả

In [ ]:
# ============================================================
# Ô 1: MOUNT DRIVE + CLONE CODE VÀO SSD
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import os

# --- Đường dẫn dữ liệu gốc trên Google Drive (CHỈ ĐỌC) ---
DRIVE_PROJECT = '/content/drive/MyDrive/Colab Notebooks/Text2Pandas'
DRIVE_DATA    = os.path.join(DRIVE_PROJECT, 'data')

# --- Clone code vào SSD (tốc độ cao, không bị permission error) ---
CODE_DIR = '/content/Code_Moi'
if os.path.exists(CODE_DIR):
    %cd {CODE_DIR}
    !git pull
else:
    %cd /content
    !git clone https://github.com/HoangKhang226/AI-Financial-Data-Assistant.git Code_Moi
    %cd {CODE_DIR}

print(f'\n\u2705 Code: {os.getcwd()}')
print(f'\u2705 Drive Data: {DRIVE_DATA}')

In [ ]:
# ============================================================
# Ô 2: CÀI ĐẶT THƯ VIỆN (giải quyết xung đột transformers + torchaudio)
# ============================================================
# Bước 2a: Gỡ torchaudio/torchvision gây xung đột với vLLM
!pip uninstall -y torchaudio torchvision 2>/dev/null

# Bước 2b: Cài từ requirements.txt (transformers==4.45.0 đã được pin)
!pip install -r requirements.txt -q

print('\n\u2705 Cài đặt thư viện hoàn tất!')

In [ ]:
# ============================================================
# Ô 3: KIỂM TRA GPU + MÔI TRƯỜNG
# ============================================================
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')
else:
    print('\u274c KHÔNG CÓ GPU! Vào Runtime > Change runtime type > T4 GPU')

import transformers; print(f'Transformers: {transformers.__version__}')
import vllm; print(f'vLLM: {vllm.__version__}')
print('\n\u2705 Môi trường sẵn sàng!')

In [ ]:
# ============================================================
# Ô 4: PIPELINE 1 - INGESTION (CHỈ CHẠY 1 LẦN)
# Đọc từ Drive -> Ghi CSV/Metadata/Index lên SSD siêu nhanh
# ============================================================
import os

# Tạo thư mục output trên SSD
SSD_DATA = '/content/data_output'
os.makedirs(f'{SSD_DATA}/csv_warehouse', exist_ok=True)
os.makedirs(f'{SSD_DATA}/metadata', exist_ok=True)
os.makedirs(f'{SSD_DATA}/index', exist_ok=True)

# Kiểm tra nếu đã chạy Pipeline 1 rồi (có file summaries.jsonl)
summ_path = f'{SSD_DATA}/index/summaries.jsonl'
if os.path.exists(summ_path) and os.path.getsize(summ_path) > 1000:
    print(f'\u2705 Pipeline 1 đã chạy rồi! ({os.path.getsize(summ_path):,} bytes)')
    print('Bỏ qua ô này, chuyển sang ô 5.')
else:
    DRIVE_DATA = '/content/drive/MyDrive/Colab Notebooks/Text2Pandas/data'
    INPUT_DIR = os.path.join(DRIVE_DATA, 'ViFinQA/financial_statements')
    
    !python -m src.pipeline_1_ingestion.pipeline \
        --input "{INPUT_DIR}" \
        --csv-out "{SSD_DATA}/csv_warehouse" \
        --meta-out "{SSD_DATA}/metadata" \
        --summ-out "{SSD_DATA}/index/summaries.jsonl"
    
    # Đếm kết quả
    n_csv = len(os.listdir(f'{SSD_DATA}/csv_warehouse')) if os.path.isdir(f'{SSD_DATA}/csv_warehouse') else 0
    n_meta = len(os.listdir(f'{SSD_DATA}/metadata')) if os.path.isdir(f'{SSD_DATA}/metadata') else 0
    print(f'\n\u2705 Pipeline 1 hoàn tất: {n_csv} CSV, {n_meta} metadata files')

In [ ]:
# ============================================================
# Ô 5: PIPELINE 2 - QUERY + LLM (TỰ RESUME QUA CHECKPOINT)
# Chạy lại bao nhiêu lần cũng được, nó tự bỏ qua câu đã làm
# ============================================================
import os

DRIVE_DATA = '/content/drive/MyDrive/Colab Notebooks/Text2Pandas/data'
SSD_DATA   = '/content/data_output'

# Tạo thư mục output
os.makedirs('/content/output', exist_ok=True)

!python -m src.pipeline_2_query.pipeline \
    --questions "{DRIVE_DATA}/test_questions.jsonl" \
    --code-stock "{DRIVE_DATA}/ViFinQA/code_stock.csv" \
    --metadata "{SSD_DATA}/metadata" \
    --csv-warehouse "{SSD_DATA}/csv_warehouse" \
    --index "{SSD_DATA}/index" \
    --output "/content/output/submission.json" \
    --checkpoint "/content/output/checkpoint_results.jsonl" \
    --use-llm --verbose

In [ ]:
# ============================================================
# Ô 6: KIỂM TRA KẾT QUẢ + SAO LƯU VỀ DRIVE
# ============================================================
import json, os, shutil

# Đếm checkpoint
chk = '/content/output/checkpoint_results.jsonl'
if os.path.exists(chk):
    with open(chk) as f:
        lines = [l for l in f if l.strip()]
    ok = sum(1 for l in lines if json.loads(l).get('success'))
    print(f'Checkpoint: {len(lines)} câu ({ok} thành công, {len(lines)-ok} thất bại)')

# Xem submission
sub_path = '/content/output/submission.json'
if os.path.exists(sub_path):
    with open(sub_path) as f:
        sub = json.load(f)
    print(f'\nSubmission: {len(sub)} câu trả lời')
    for k, v in list(sub.items())[:5]:
        print(f'  Q{k}: {v}')
    
    # Sao lưu về Drive
    DRIVE_OUT = '/content/drive/MyDrive/Colab Notebooks/Text2Pandas/output'
    os.makedirs(DRIVE_OUT, exist_ok=True)
    shutil.copy2(sub_path, os.path.join(DRIVE_OUT, 'submission.json'))
    print(f'\n\u2705 Đã sao lưu submission.json về Google Drive!')
else:
    print('Chưa có submission! Hãy chạy ô 5 trước.')